In [ ]:
import os
import sys
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, isnan, min, max, mean, avg, stddev,
    lit, hour, dayofweek, month, to_date, desc, asc, round as spark_round,
    unix_timestamp, expr
)
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .appName("TrafficProject")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")


In [ ]:
data_path = "data/traffic_data.csv" if os.path.exists("data/traffic_data.csv") else "../data/traffic_data.csv"
output_parquet = "output/traffic_cleaned.parquet" if os.path.exists("output") else "../output/traffic_cleaned.parquet"

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(data_path)
)
df_raw.printSchema()
df_raw.show(5, truncate=False)


In [ ]:
print("Row count:", df_raw.count())
print("Column count:", len(df_raw.columns))

missing_exprs = []
for field in df_raw.schema.fields:
    c = col(field.name)
    if field.dataType.typeName() in ("double", "float"):
        missing_exprs.append(count(when(c.isNull() | isnan(c), 1)).alias(field.name))
    else:
        missing_exprs.append(count(when(c.isNull(), 1)).alias(field.name))

df_raw.select(*missing_exprs).show()

duplicate_rows = df_raw.count() - df_raw.dropDuplicates().count()
print("Exact duplicate rows:", duplicate_rows)

df_raw.describe("temp", "rain_1h", "snow_1h", "clouds_all", "traffic_volume").show()

df_raw.select(
    count("*").alias("total"),
    count(when(col("snow_1h") == 0, 1)).alias("zero_snow"),
    count(when(col("snow_1h") > 0, 1)).alias("pos_snow"),
    max("snow_1h").alias("max_snow")
).show()

df_raw.select(
    count(when(col("traffic_volume") < 0, 1)).alias("neg_traffic"),
    count(when(col("rain_1h") < 0, 1)).alias("neg_rain"),
    count(when(col("snow_1h") < 0, 1)).alias("neg_snow"),
    count(when(col("clouds_all") < 0, 1)).alias("neg_clouds"),
    count(when(col("clouds_all") > 100, 1)).alias("clouds_gt_100")
).show()


In [ ]:
df_raw.filter(col("temp") == 0).show(10, truncate=False)

df_raw.orderBy(col("rain_1h").desc()).select(
    "date_time", "temp", "rain_1h", "snow_1h", "weather_main", "weather_description", "traffic_volume"
).show(10, truncate=False)

df_raw.filter(col("traffic_volume") == 0).select(
    "date_time", "temp", "rain_1h", "snow_1h", "weather_main", "traffic_volume"
).show()

df_raw.groupBy(*df_raw.columns).agg(count("*").alias("copies")).filter(col("copies") > 1).show(10, truncate=False)

df_raw.select(
    min("date_time").alias("min_time"),
    max("date_time").alias("max_time"),
    count("*").alias("total")
).show()


In [ ]:
df_dedup = df_raw.dropDuplicates()

df_sanitized = df_dedup.withColumn(
    "temp",
    when(col("temp") == 0, lit(None).cast("double"))
    .otherwise(col("temp"))
).withColumn(
    "rain_1h",
    when(col("rain_1h") > 100, lit(None).cast("double"))
    .otherwise(col("rain_1h"))
)

df_cleaned = (
    df_sanitized
    .withColumn("date", to_date("date_time"))
    .withColumn("hour", hour("date_time"))
    .withColumn("day_of_week", dayofweek("date_time"))
    .withColumn("month", month("date_time"))
    .withColumn(
        "is_weekend",
        when(dayofweek("date_time").isin(1, 7), 1)
        .otherwise(0)
    )
)

df_cleaned.select(
    count("*").alias("final_rows"),
    count(when(col("temp").isNull(), 1)).alias("temp_nulls"),
    count(when(col("rain_1h").isNull(), 1)).alias("rain_nulls"),
    count(when(col("traffic_volume") == 0, 1)).alias("zero_traffic"),
    count(when(col("snow_1h") == 0, 1)).alias("zero_snow")
).show()

os.makedirs(os.path.dirname(output_parquet) or ".", exist_ok=True)
df_cleaned.write.mode("overwrite").parquet(output_parquet)


In [ ]:
df_cleaned.createOrReplaceTempView("traffic")

hourly_df = spark.sql('''
SELECT 
    hour,
    ROUND(AVG(traffic_volume), 2) AS avg_traffic,
    ROUND(STDDEV(traffic_volume), 2) AS std_traffic,
    MIN(traffic_volume) AS min_traffic,
    MAX(traffic_volume) AS max_traffic,
    COUNT(*) AS count
FROM traffic
GROUP BY hour
ORDER BY hour
''')
hourly_df.show(24, truncate=False)

weekday_weekend_df = spark.sql('''
SELECT 
    hour,
    is_weekend,
    ROUND(AVG(traffic_volume), 2) AS avg_traffic,
    COUNT(*) AS count
FROM traffic
GROUP BY hour, is_weekend
ORDER BY hour, is_weekend
''')
weekday_weekend_df.show(10, truncate=False)

monthly_df = spark.sql('''
SELECT 
    month,
    ROUND(AVG(traffic_volume), 2) AS avg_traffic,
    ROUND(AVG(temp), 2) AS avg_temp,
    ROUND(AVG(rain_1h), 3) AS avg_rain,
    COUNT(*) AS count
FROM traffic
GROUP BY month
ORDER BY month
''')
monthly_df.show(12, truncate=False)

weather_df = spark.sql('''
SELECT 
    weather_main,
    COUNT(*) AS count,
    ROUND(AVG(traffic_volume), 2) AS avg_traffic,
    ROUND(AVG(temp), 2) AS avg_temp,
    ROUND(AVG(rain_1h), 3) AS avg_rain
FROM traffic
GROUP BY weather_main
ORDER BY avg_traffic DESC
''')
weather_df.show(truncate=False)

holiday_df = spark.sql('''
SELECT 
    CASE WHEN holiday IS NULL OR holiday = 'None' THEN 'Regular Day' ELSE 'Holiday' END AS status,
    COUNT(*) AS count,
    ROUND(AVG(traffic_volume), 2) AS avg_traffic,
    MIN(traffic_volume) AS min_traffic,
    MAX(traffic_volume) AS max_traffic
FROM traffic
GROUP BY CASE WHEN holiday IS NULL OR holiday = 'None' THEN 'Regular Day' ELSE 'Holiday' END
''')
holiday_df.show(truncate=False)


In [ ]:
cutoff_ts = df_cleaned.select(expr("percentile_approx(unix_timestamp(date_time), 0.80)")).collect()[0][0]

train_df = df_cleaned.filter(unix_timestamp("date_time") <= cutoff_ts).cache()
test_df = df_cleaned.filter(unix_timestamp("date_time") > cutoff_ts).cache()

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

imputer = Imputer(
    inputCols=["temp", "rain_1h"],
    outputCols=["temp_imputed", "rain_1h_imputed"],
    strategy="median"
)

weather_indexer = StringIndexer(inputCol="weather_main", outputCol="weather_main_idx", handleInvalid="keep")
holiday_indexer = StringIndexer(inputCol="holiday", outputCol="holiday_idx", handleInvalid="keep")

encoder = OneHotEncoder(
    inputCols=["weather_main_idx", "holiday_idx"],
    outputCols=["weather_main_vec", "holiday_vec"],
    handleInvalid="keep"
)

feature_cols = [
    "hour", "day_of_week", "month", "is_weekend",
    "temp_imputed", "rain_1h_imputed", "snow_1h", "clouds_all",
    "weather_main_vec", "holiday_vec"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

lr = LinearRegression(featuresCol="features", labelCol="traffic_volume", maxIter=50, regParam=0.1)
rf = RandomForestRegressor(featuresCol="features", labelCol="traffic_volume", numTrees=60, maxDepth=12, seed=42)

lr_pipeline = Pipeline(stages=[imputer, weather_indexer, holiday_indexer, encoder, assembler, lr])
rf_pipeline = Pipeline(stages=[imputer, weather_indexer, holiday_indexer, encoder, assembler, rf])

lr_model = lr_pipeline.fit(train_df)
rf_model = rf_pipeline.fit(train_df)

evaluator_rmse = RegressionEvaluator(labelCol="traffic_volume", predictionCol="prediction", metricName="rmse")
evaluator_mae = RegressionEvaluator(labelCol="traffic_volume", predictionCol="prediction", metricName="mae")
evaluator_r2 = RegressionEvaluator(labelCol="traffic_volume", predictionCol="prediction", metricName="r2")

lr_preds = lr_model.transform(test_df)
rf_preds = rf_model.transform(test_df)

print("Linear Regression -> RMSE:", evaluator_rmse.evaluate(lr_preds), "MAE:", evaluator_mae.evaluate(lr_preds), "R2:", evaluator_r2.evaluate(lr_preds))
print("Random Forest     -> RMSE:", evaluator_rmse.evaluate(rf_preds), "MAE:", evaluator_mae.evaluate(rf_preds), "R2:", evaluator_r2.evaluate(rf_preds))

model_save_path = "output/models/rf_traffic_pipeline" if os.path.exists("output") else "../output/models/rf_traffic_pipeline"
rf_model.write().overwrite().save(model_save_path)


In [ ]:
loaded_model = PipelineModel.load(model_save_path)

sample_input = spark.createDataFrame([{
    "hour": 8,
    "day_of_week": 3,
    "month": 10,
    "is_weekend": 0,
    "temp": 288.15,
    "rain_1h": 0.0,
    "snow_1h": 0.0,
    "clouds_all": 40,
    "weather_main": "Clouds",
    "holiday": "None"
}])

pred = loaded_model.transform(sample_input).select("prediction").collect()[0][0]
print("Predicted volume:", round(pred, 1))


In [ ]:
spark.stop()
